# Conditional Edges & Routing

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) LangGraph roadmap.

You will learn how to use `add_conditional_edges` for runtime branching, LLM-based routing with `with_structured_output`, and how to route to specialized generator nodes.

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain "langchain[openai]"

## 2. Set Up Your API Key

Enter your OpenAI API key when prompted. The key is not stored or displayed.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Simple Keyword-Based Routing

The simplest form of conditional routing uses a function that inspects the state and returns a string key.

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END, add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]
    category: str

def classify(state: State) -> dict:
    last = state["messages"][-1].content.lower()
    if "poem" in last:
        return {"category": "poem"}
    elif "joke" in last:
        return {"category": "joke"}
    return {"category": "story"}

def route(state: State) -> str:
    return state["category"]

def poem_node(state: State) -> dict:
    return {"messages": [("ai", "Here is a poem for you...")]}

def joke_node(state: State) -> dict:
    return {"messages": [("ai", "Here is a joke for you...")]}

def story_node(state: State) -> dict:
    return {"messages": [("ai", "Here is a story for you...")]}

graph = StateGraph(State)
graph.add_node("classify", classify)
graph.add_node("poem", poem_node)
graph.add_node("joke", joke_node)
graph.add_node("story", story_node)

graph.add_edge(START, "classify")
graph.add_conditional_edges("classify", route, {
    "poem": "poem",
    "joke": "joke",
    "story": "story",
})
graph.add_edge("poem", END)
graph.add_edge("joke", END)
graph.add_edge("story", END)

app = graph.compile()

result = app.invoke({"messages": [("human", "Tell me a joke about dogs")]})
print(f"Category: {result['category']}")
print(f"Response: {result['messages'][-1].content}")

## 4. LLM-Based Routing with Structured Output

Use `with_structured_output` and a Pydantic model with `Literal` types for robust classification.

In [ ]:
from pydantic import BaseModel
from typing import Literal
from langchain_openai import ChatOpenAI

class RouteModel(BaseModel):
    """Classify the user request into one of the allowed categories."""
    category: Literal["poem", "joke", "story"]

llm = ChatOpenAI(model="gpt-4o-mini")
classifier_llm = llm.with_structured_output(RouteModel)

def llm_classify(state: State) -> dict:
    result = classifier_llm.invoke(state["messages"])
    return {"category": result.category}

test_result = classifier_llm.invoke([("human", "Write me a haiku about rain")])
print(f"Classified as: {test_result.category}")

## 5. Full Router with LLM Classification and Generators

Combine LLM-based classification with specialized generator nodes.

In [ ]:
def poem_generator(state: State) -> dict:
    response = llm.invoke([
        ("system", "You are a poet. Write a short poem based on the user's request."),
        *state["messages"],
    ])
    return {"messages": [response]}

def joke_generator(state: State) -> dict:
    response = llm.invoke([
        ("system", "You are a comedian. Write a short joke based on the user's request."),
        *state["messages"],
    ])
    return {"messages": [response]}

def story_generator(state: State) -> dict:
    response = llm.invoke([
        ("system", "You are a storyteller. Write a short story based on the user's request."),
        *state["messages"],
    ])
    return {"messages": [response]}

graph = StateGraph(State)
graph.add_node("classify", llm_classify)
graph.add_node("poem", poem_generator)
graph.add_node("joke", joke_generator)
graph.add_node("story", story_generator)

graph.add_edge(START, "classify")
graph.add_conditional_edges("classify", route, {
    "poem": "poem",
    "joke": "joke",
    "story": "story",
})
graph.add_edge("poem", END)
graph.add_edge("joke", END)
graph.add_edge("story", END)

app = graph.compile()

result = app.invoke({"messages": [("human", "Tell me a funny joke about cats")]})
print(f"Category: {result['category']}")
print(f"Output:\n{result['messages'][-1].content}")

In [ ]:
result = app.invoke({"messages": [("human", "Write a poem about the stars")]})
print(f"Category: {result['category']}")
print(f"Output:\n{result['messages'][-1].content}")

## What You Learned

1. **`add_conditional_edges(source, routing_fn, mapping)`** enables runtime branching based on state
2. **`with_structured_output`** with `Literal` types ensures the LLM returns valid routing keys
3. **Routing functions** inspect state and return a string key matching the mapping dict
4. **Generator nodes** are regular nodes with specialized system prompts
5. Keep the `Literal` values and mapping dict in sync when adding new routes